# STAIR-v1 — Stepwise Graph Contrastive Learning (STAIR-GCL)

**Mô hình:** STAIR Baseline vs STAIR-v1 (STAIR-GCL)
**Ý tưởng 1 (Core & Loss):**  
1. Loại bỏ cạnh ngẫu nhiên (Edge Dropout $p_{drop}=0.2$) tạo 2 view đồ thị phụ trợ $\mathcal{G}'$ và $\mathcal{G}''$.  
2. Áp dụng InfoNCE Contrastive Loss **chỉ cho 32 chiều đầu** (Collaborative Subspace), giúp lọc nhiễu tương tác.  
3. Giữ nguyên 32 chiều sau (Multimodal Subspace) bảo tồn thuộc tính hình ảnh/văn bản qua FSC không bị oversmoothing.

**Datasets thử nghiệm:** `baby` (Amazon2014Baby_550_MMRec) & `sports` (Amazon2014Sports_550_MMRec)

## Cell 1 — Thiết lập Môi trường & Kéo Mã nguồn Public từ GitHub

In [ ]:
import os, shutil

os.chdir('/kaggle/working')

github_user = 'ThanhChuong12'
repo_name   = 'STAIR-Enhanced'

if os.path.exists(repo_name):
    shutil.rmtree(repo_name)
    print('Da xoa thu muc ' + repo_name + ' cu.')

git_url = 'https://github.com/' + github_user + '/' + repo_name + '.git'
print('Dang clone public repo: ' + git_url)
os.system('git clone ' + git_url)

print('Dang cai dat thu vien freerec, torch_geometric, nvidia-ml-py...')
os.system('pip install nvidia-ml-py -q')
os.system('pip install torchdata==0.6.1 --no-deps -q')
os.system('pip install freerec==0.8.3 -q')
os.system('pip install torch_geometric -q')

os.chdir('/kaggle/working/' + repo_name)
print('Thu muc hien tai: ' + os.getcwd())
for f in ['main.py', 'main_v1.py']:
    exists = os.path.exists(f)
    print('  [' + ('OK' if exists else 'MISSING!') + '] ' + f)
print('Moi truong va repo da san sang!')

## Cell 2 — Chuẩn bị Dữ liệu (`baby` & `sports`) vào `/kaggle/data`

In [ ]:
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

dataset_map = {
    'Amazon2014Baby_550_MMRec':   '/kaggle/input/stair-datasets-mmrec/Amazon2014Baby_550_MMRec',
    'Amazon2014Sports_550_MMRec': '/kaggle/input/stair-datasets-mmrec/Amazon2014Sports_550_MMRec',
}

print('Copy du lieu vao ' + DATA_ROOT + '...')
for ds_name, src in dataset_map.items():
    dest = os.path.join(DATA_ROOT, ds_name)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    if os.path.exists(src):
        shutil.copytree(src, dest)
        files = os.listdir(dest)
        print('  [OK] ' + ds_name + ': ' + str(files))
    else:
        # Thu fallback tu tim trong /kaggle/input
        found = False
        for root, dirs, _ in os.walk('/kaggle/input'):
            if ds_name.lower() in root.lower():
                shutil.copytree(root, dest)
                print('  [OK Fallback] ' + ds_name + ': ' + str(os.listdir(dest)))
                found = True
                break
        if not found:
            print('  [ERR] Khong tim thay dataset: ' + ds_name)

print('Du lieu da san sang!')

## Cell 3 — Kiểm tra GPU & Môi trường

In [ ]:
import torch, platform, subprocess, os

print('='*60)
print('THONG TIN MOI TRUONG')
print('='*60)
print('  Python     : ' + platform.python_version())
print('  PyTorch    : ' + torch.__version__)
print('  CUDA avail : ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    print('  GPU name   : ' + torch.cuda.get_device_name(0))
    cap = torch.cuda.get_device_capability(0)
    print('  Capability : ' + str(cap[0]) + '.' + str(cap[1]))
print('  Working dir: ' + os.getcwd())
print('='*60)

## Cell 4 — Huấn luyện Baseline vs STAIR-v1 (STAIR-GCL) trên `baby` & `sports`

> Chạy huấn luyện 500 epochs cho cả Baseline (`main.py`) và STAIR-v1 (`main_v1.py`).

In [ ]:
import pynvml, time, threading, subprocess, os

runs = [
    # --- BABY ---
    {
        'key': 'baby_baseline',
        'script': 'main.py',
        'dataset': 'Amazon2014Baby_550_MMRec',
        'weight_decay': '0.3',
        'gamma': '0.1',
        'extra_args': []
    },
    {
        'key': 'baby_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Baby_550_MMRec',
        'weight_decay': '0.3',
        'gamma': '0.1',
        'extra_args': ['--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
    # --- SPORTS ---
    {
        'key': 'sports_baseline',
        'script': 'main.py',
        'dataset': 'Amazon2014Sports_550_MMRec',
        'weight_decay': '0.1',
        'gamma': '0.2',
        'extra_args': []
    },
    {
        'key': 'sports_v1',
        'script': 'main_v1.py',
        'dataset': 'Amazon2014Sports_550_MMRec',
        'weight_decay': '0.1',
        'gamma': '0.2',
        'extra_args': ['--cl-weight', '1e-3', '--edge-drop', '0.2', '--cl-temp', '0.2']
    },
]

results_log = {}

for run in runs:
    key = run['key']
    script = run['script']
    dataset = run['dataset']
    print('\n' + '='*60)
    print('🚀 DANG HUAN LUYEN: ' + key.upper() + ' (' + script + ')')
    print('='*60)

    cmd = [
        'python', script,
        '--dataset',      dataset,
        '--root',         '/kaggle/data',
        '--weight-decay', run['weight_decay'],
        '--gamma',        run['gamma'],
        '--epochs',       '500',
        '--eval-freq',    '5',
    ] + run['extra_args']

    print('Lenh: ' + ' '.join(cmd))
    start_time = time.time()
    res = subprocess.run(cmd, capture_output=True, text=True, cwd='/kaggle/working/STAIR-Enhanced')
    elapsed = time.time() - start_time

    if res.returncode != 0:
        print('❌ LOI ' + key.upper() + ':')
        print(res.stderr[-3000:])
        results_log[key] = {'status': 'FAIL', 'log': res.stderr[-2000:], 'time': elapsed}
    else:
        print('✅ HOAN THANH ' + key.upper() + ' trong ' + str(round(elapsed, 1)) + 's!')
        print(res.stdout[-1500:])
        results_log[key] = {'status': 'OK', 'log': res.stdout, 'time': elapsed}

print('\n' + '='*60)
print('TAT CA PHIEN RUN DA HOAN THANH!')
print('='*60)

## Cell 5 — Trích xuất Metrics & Vẽ Biểu Đồ So Sánh Baseline vs STAIR-v1

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np

# Metrics extractor from FreeRec stdout
def extract_best_metrics(stdout_text):
    metrics = {}
    pattern = re.compile(r'(Recall@\d+|NDCG@\d+)\s*:\s*([0-9.]+)')
    for line in stdout_text.splitlines():
        if 'Best' in line or 'test' in line.lower() or 'eval' in line.lower() or 'Result' in line:
            for match in pattern.finditer(line):
                metrics[match.group(1)] = float(match.group(2))
    # Fallback search anywhere
    if not metrics:
        for match in pattern.finditer(stdout_text):
            metrics[match.group(1)] = float(match.group(2))
    return metrics

summary_metrics = {}
for key, item in results_log.items():
    if item['status'] == 'OK':
        m = extract_best_metrics(item['log'])
        summary_metrics[key] = m
        print(f"{key.upper()}: {m}")
    else:
        print(f"{key.upper()}: FAILED")

# Plotting comparison bar charts
datasets = ['baby', 'sports']
metric_keys = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('STAIR Baseline vs STAIR-v1 (STAIR-GCL) Performance Comparison', fontsize=15, fontweight='bold')

for idx, ds in enumerate(datasets):
    ax = axes[idx]
    base_key = f"{ds}_baseline"
    v1_key   = f"{ds}_v1"
    
    base_vals = [summary_metrics.get(base_key, {}).get(m, 0.0) for m in metric_keys]
    v1_vals   = [summary_metrics.get(v1_key, {}).get(m, 0.0) for m in metric_keys]
    
    x = np.arange(len(metric_keys))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, base_vals, width, label='STAIR Baseline', color='#3498DB', alpha=0.85)
    bars2 = ax.bar(x + width/2, v1_vals, width, label='STAIR-v1 (GCL)', color='#2ECC71', alpha=0.85)
    
    for bar in bars1 + bars2:
        h = bar.get_height()
        if h > 0:
            ax.annotate(f"{h:.4f}", xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
    
    ax.set_title(f"Dataset: {ds.upper()}", fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_keys, fontsize=10)
    ax.set_ylabel('Metric Value')
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('/kaggle/working/stair_v1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved chart to: /kaggle/working/stair_v1_comparison.png')